## **01. Subconjunto No Dominado**

### **01.1 Script para Calcular Conjunto de Alternativas No Dominadas**

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from functools import wraps
from time import perf_counter
from typing import Any, Callable, Sequence

import numpy as np
import numpy.typing as npt


class Direction(str, Enum):
    MIN = "min"
    MAX = "max"


@dataclass(frozen=True, slots=True)
class DominanceRelation:
    dominated_index: int
    dominated_alternative: str
    dominator_index: int
    dominator_alternative: str

    def to_dict(self) -> dict[str, Any]:
        return {
            "dominated_index": self.dominated_index,
            "dominated_alternative": self.dominated_alternative,
            "dominator_index": self.dominator_index,
            "dominator_alternative": self.dominator_alternative,
        }


@dataclass(frozen=True, slots=True)
class ComparisonRecord:
    comparison_number: int

    reference_index: int
    reference_alternative: str

    candidate_index: int
    candidate_alternative: str

    dominates: bool
    better_or_equal_all: bool
    strictly_better_at_least_one: bool

    better_dimensions: list[str]
    equal_dimensions: list[str]
    worse_dimensions: list[str]

    def to_dict(self) -> dict[str, Any]:
        return {
            "comparison_number": self.comparison_number,
            "reference_index": self.reference_index,
            "reference_alternative": self.reference_alternative,
            "candidate_index": self.candidate_index,
            "candidate_alternative": self.candidate_alternative,
            "dominates": self.dominates,
            "better_or_equal_all": self.better_or_equal_all,
            "strictly_better_at_least_one": self.strictly_better_at_least_one,
            "better_dimensions": self.better_dimensions,
            "equal_dimensions": self.equal_dimensions,
            "worse_dimensions": self.worse_dimensions,
        }


@dataclass(frozen=True, slots=True)
class ComparisonStats:
    alternative_pair_comparisons: int
    dimension_comparisons_better_or_equal: int
    dimension_comparisons_strictly_better: int
    total_dimension_comparisons: int
    alternatives_evaluated: int

    def to_dict(self) -> dict[str, int]:
        return {
            "alternative_pair_comparisons": self.alternative_pair_comparisons,
            "dimension_comparisons_better_or_equal": (
                self.dimension_comparisons_better_or_equal
            ),
            "dimension_comparisons_strictly_better": (
                self.dimension_comparisons_strictly_better
            ),
            "total_dimension_comparisons": self.total_dimension_comparisons,
            "alternatives_evaluated": self.alternatives_evaluated,
        }


@dataclass(frozen=True, slots=True)
class ParetoResult:
    pareto_indices: list[int]
    pareto_alternatives: list[str]

    dominated_indices: list[int]
    dominated_alternatives: list[str]

    pareto_mask: npt.NDArray[np.bool_]
    dominated_mask: npt.NDArray[np.bool_]

    dominance_relations: list[DominanceRelation]

    comparison_stats: ComparisonStats
    comparison_trace: list[ComparisonRecord]

    transformed_matrix: npt.NDArray[np.float64]
    execution_time_seconds: float


def timed_method(
    method: Callable[..., ParetoResult],
) -> Callable[..., ParetoResult]:
    @wraps(method)
    def wrapper(self: "ParetoSolver", *args: Any, **kwargs: Any) -> ParetoResult:
        start = perf_counter()
        result = method(self, *args, **kwargs)
        end = perf_counter()

        return ParetoResult(
            pareto_indices=result.pareto_indices,
            pareto_alternatives=result.pareto_alternatives,
            dominated_indices=result.dominated_indices,
            dominated_alternatives=result.dominated_alternatives,
            pareto_mask=result.pareto_mask,
            dominated_mask=result.dominated_mask,
            dominance_relations=result.dominance_relations,
            comparison_stats=result.comparison_stats,
            comparison_trace=result.comparison_trace,
            transformed_matrix=result.transformed_matrix,
            execution_time_seconds=end - start,
        )

    return wrapper


class ParetoSolver:
    def __init__(
        self,
        matrix: Sequence[Sequence[float]] | npt.NDArray[np.float64],
        dimensions: Sequence[str],
        directions: Sequence[str | Direction],
        alternatives: Sequence[str],
        *,
        epsilon: float = 1e-12,
    ) -> None:
        self._raw_matrix = np.asarray(matrix, dtype=np.float64)
        self._dimensions = list(dimensions)
        self._directions = [self._parse_direction(direction) for direction in directions]
        self._alternatives = [str(alternative) for alternative in alternatives]
        self._epsilon = float(epsilon)

        self._validate_inputs()

        self._n_alternatives: int = self._raw_matrix.shape[0]
        self._n_dimensions: int = self._raw_matrix.shape[1]

        self._sign_vector = self._build_sign_vector()
        self._transformed_matrix = self._raw_matrix * self._sign_vector

    @property
    def matrix(self) -> npt.NDArray[np.float64]:
        return self._raw_matrix.copy()

    @property
    def transformed_matrix(self) -> npt.NDArray[np.float64]:
        return self._transformed_matrix.copy()

    @property
    def dimensions(self) -> list[str]:
        return self._dimensions.copy()

    @property
    def directions(self) -> list[Direction]:
        return self._directions.copy()

    @property
    def alternatives(self) -> list[str]:
        return self._alternatives.copy()

    @staticmethod
    def _parse_direction(direction: str | Direction) -> Direction:
        if isinstance(direction, Direction):
            return direction

        normalized = str(direction).strip().lower()

        aliases = {
            "min": Direction.MIN,
            "minimize": Direction.MIN,
            "minimizar": Direction.MIN,
            "minimum": Direction.MIN,
            "max": Direction.MAX,
            "maximize": Direction.MAX,
            "maximizar": Direction.MAX,
            "maximum": Direction.MAX,
        }

        if normalized not in aliases:
            raise ValueError(
                f"Invalid direction '{direction}'. "
                "Use 'min'/'max', 'minimizar'/'maximizar', or equivalent aliases."
            )

        return aliases[normalized]

    def _validate_inputs(self) -> None:
        if self._raw_matrix.ndim != 2:
            raise ValueError("matrix must be a two-dimensional array.")

        if self._raw_matrix.size == 0:
            raise ValueError("matrix cannot be empty.")

        if not np.all(np.isfinite(self._raw_matrix)):
            raise ValueError("matrix contains NaN or infinite values.")

        n_alternatives, n_dimensions = self._raw_matrix.shape

        if len(self._dimensions) != n_dimensions:
            raise ValueError(
                "The number of dimensions must match the number of matrix columns. "
                f"Expected {n_dimensions}, got {len(self._dimensions)}."
            )

        if len(self._directions) != n_dimensions:
            raise ValueError(
                "The number of directions must match the number of matrix columns. "
                f"Expected {n_dimensions}, got {len(self._directions)}."
            )

        if len(self._alternatives) != n_alternatives:
            raise ValueError(
                "The number of alternatives must match the number of matrix rows. "
                f"Expected {n_alternatives}, got {len(self._alternatives)}."
            )

        if len(set(self._dimensions)) != len(self._dimensions):
            raise ValueError("Dimension names must be unique.")

        if len(set(self._alternatives)) != len(self._alternatives):
            raise ValueError("Alternative identifiers must be unique.")

        if self._epsilon < 0:
            raise ValueError("epsilon must be greater than or equal to zero.")

    def _build_sign_vector(self) -> npt.NDArray[np.float64]:
        return np.array(
            [
                1.0 if direction == Direction.MAX else -1.0
                for direction in self._directions
            ],
            dtype=np.float64,
        )

    def _dominates_fast(
        self,
        *,
        candidate_index: int,
        reference_index: int,
    ) -> bool:
        candidate = self._transformed_matrix[candidate_index]
        reference = self._transformed_matrix[reference_index]

        better_or_equal_all = np.all(candidate >= reference - self._epsilon)
        strictly_better_one = np.any(candidate > reference + self._epsilon)

        return bool(better_or_equal_all and strictly_better_one)

    def _compare_pair_with_trace(
        self,
        *,
        candidate_index: int,
        reference_index: int,
        comparison_number: int,
    ) -> tuple[bool, ComparisonRecord]:
        candidate = self._transformed_matrix[candidate_index]
        reference = self._transformed_matrix[reference_index]

        better_or_equal_mask = candidate >= reference - self._epsilon
        strictly_better_mask = candidate > reference + self._epsilon
        worse_mask = candidate < reference - self._epsilon
        equal_mask = ~(strictly_better_mask | worse_mask)

        better_or_equal_all = bool(np.all(better_or_equal_mask))
        strictly_better_at_least_one = bool(np.any(strictly_better_mask))

        dominates = better_or_equal_all and strictly_better_at_least_one

        record = ComparisonRecord(
            comparison_number=comparison_number,
            reference_index=reference_index,
            reference_alternative=self._alternatives[reference_index],
            candidate_index=candidate_index,
            candidate_alternative=self._alternatives[candidate_index],
            dominates=dominates,
            better_or_equal_all=better_or_equal_all,
            strictly_better_at_least_one=strictly_better_at_least_one,
            better_dimensions=[
                self._dimensions[i]
                for i, is_better in enumerate(strictly_better_mask)
                if is_better
            ],
            equal_dimensions=[
                self._dimensions[i]
                for i, is_equal in enumerate(equal_mask)
                if is_equal
            ],
            worse_dimensions=[
                self._dimensions[i]
                for i, is_worse in enumerate(worse_mask)
                if is_worse
            ],
        )

        return dominates, record

    @timed_method
    def solve(
        self,
        *,
        collect_all_dominators: bool = False,
        trace: bool = False,
    ) -> ParetoResult:
        n = self._n_alternatives
        m = self._n_dimensions

        dominated_mask = np.zeros(n, dtype=np.bool_)
        dominance_relations: list[DominanceRelation] = []
        comparison_trace: list[ComparisonRecord] = []

        alternative_pair_comparisons = 0
        alternatives_evaluated = 0

        for reference_index in range(n):
            if dominated_mask[reference_index] and not collect_all_dominators:
                continue

            alternatives_evaluated += 1

            for candidate_index in range(n):
                if candidate_index == reference_index:
                    continue

                alternative_pair_comparisons += 1

                if trace:
                    dominates, record = self._compare_pair_with_trace(
                        candidate_index=candidate_index,
                        reference_index=reference_index,
                        comparison_number=alternative_pair_comparisons,
                    )
                    comparison_trace.append(record)
                else:
                    dominates = self._dominates_fast(
                        candidate_index=candidate_index,
                        reference_index=reference_index,
                    )

                if dominates:
                    dominated_mask[reference_index] = True

                    dominance_relations.append(
                        DominanceRelation(
                            dominated_index=reference_index,
                            dominated_alternative=self._alternatives[reference_index],
                            dominator_index=candidate_index,
                            dominator_alternative=self._alternatives[candidate_index],
                        )
                    )

                    if not collect_all_dominators:
                        break

        pareto_mask = ~dominated_mask

        pareto_indices = np.flatnonzero(pareto_mask).astype(int).tolist()
        dominated_indices = np.flatnonzero(dominated_mask).astype(int).tolist()

        pareto_alternatives = [
            self._alternatives[index]
            for index in pareto_indices
        ]

        dominated_alternatives = [
            self._alternatives[index]
            for index in dominated_indices
        ]

        dimension_comparisons_better_or_equal = alternative_pair_comparisons * m
        dimension_comparisons_strictly_better = alternative_pair_comparisons * m

        comparison_stats = ComparisonStats(
            alternative_pair_comparisons=alternative_pair_comparisons,
            dimension_comparisons_better_or_equal=dimension_comparisons_better_or_equal,
            dimension_comparisons_strictly_better=dimension_comparisons_strictly_better,
            total_dimension_comparisons=(
                dimension_comparisons_better_or_equal
                + dimension_comparisons_strictly_better
            ),
            alternatives_evaluated=alternatives_evaluated,
        )

        return ParetoResult(
            pareto_indices=pareto_indices,
            pareto_alternatives=pareto_alternatives,
            dominated_indices=dominated_indices,
            dominated_alternatives=dominated_alternatives,
            pareto_mask=pareto_mask,
            dominated_mask=dominated_mask,
            dominance_relations=dominance_relations,
            comparison_stats=comparison_stats,
            comparison_trace=comparison_trace,
            transformed_matrix=self._transformed_matrix.copy(),
            execution_time_seconds=0.0,
        )

    def solve_as_dict(
        self,
        *,
        collect_all_dominators: bool = False,
        trace: bool = False,
        include_transformed_matrix: bool = False,
        include_metadata: bool = True,
        max_dominance_relations: int | None = None,
        max_trace_records: int | None = None,
    ) -> dict[str, Any]:
        result = self.solve(
            collect_all_dominators=collect_all_dominators,
            trace=trace,
        )

        dominance_relations = result.dominance_relations
        comparison_trace = result.comparison_trace

        if max_dominance_relations is not None:
            dominance_relations = dominance_relations[:max_dominance_relations]

        if max_trace_records is not None:
            comparison_trace = comparison_trace[:max_trace_records]

        output: dict[str, Any] = {
            "pareto_indices": result.pareto_indices,
            "pareto_alternatives": result.pareto_alternatives,
            "dominated_indices": result.dominated_indices,
            "dominated_alternatives": result.dominated_alternatives,
            "pareto_mask": result.pareto_mask.tolist(),
            "dominated_mask": result.dominated_mask.tolist(),
            "dominance_relations": [
                relation.to_dict()
                for relation in dominance_relations
            ],
            "comparison_stats": result.comparison_stats.to_dict(),
            "comparison_trace": [
                record.to_dict()
                for record in comparison_trace
            ],
            "execution_time_seconds": result.execution_time_seconds,
        }

        if include_metadata:
            output["metadata"] = {
                "number_of_alternatives": self._n_alternatives,
                "number_of_dimensions": self._n_dimensions,
                "dimensions": self._dimensions,
                "directions": [direction.value for direction in self._directions],
                "epsilon": self._epsilon,
                "dominance_mode": (
                    "all_dominators"
                    if collect_all_dominators
                    else "first_dominator_only"
                ),
                "trace_enabled": trace,
                "maximum_possible_pair_comparisons": (
                    self._n_alternatives * (self._n_alternatives - 1)
                ),
                "comparison_counting_policy": (
                    "logical_dimension_evaluations_for_pareto_dominance"
                ),
            }

        if include_transformed_matrix:
            output["transformed_matrix"] = result.transformed_matrix.tolist()

        return output

### **01.2 Caso de Uso**

In [ ]:
import json

def main() -> None:
    matrix = [
        [170, 0.6393, 0.2300],
        [195, 0.7090, 0.3000],
        [128, 0.6716, 0.7000],
        [188, 0.6138, 0.8000],
        [115, 0.6442, 0.1000],
        [135, 0.5642, 0.4000],
    ]

    dimensions = ["OMOC", "OMOE", "OMOR"]
    directions = ["min", "max", "min"]

    
    alternatives = ["A", "B", "C", "D", "E", "F"]

    solver = ParetoSolver(
        matrix=matrix,
        dimensions=dimensions,
        directions=directions,
        alternatives=alternatives,
        epsilon=1e-12,
    )

    # output = solver.solve_as_dict(
    #     collect_all_dominators=False,
    #     trace=True,
    #     include_transformed_matrix=False,
    #     include_metadata=True,
    # )

    output = solver.solve_as_dict(
    collect_all_dominators=False,
    trace=True,
    include_metadata=False,
    max_dominance_relations=1,
    max_trace_records=5,
    )

    print(
        json.dumps(
            output,
            indent=4,
            ensure_ascii=False,
        )
    )


# if __name__ == "__main__":
#     main()

{
    "pareto_indices": [
        1,
        2,
        4
    ],
    "pareto_alternatives": [
        "B",
        "C",
        "E"
    ],
    "dominated_indices": [
        0,
        3,
        5
    ],
    "dominated_alternatives": [
        "A",
        "D",
        "F"
    ],
    "pareto_mask": [
        false,
        true,
        true,
        false,
        true,
        false
    ],
    "dominated_mask": [
        true,
        false,
        false,
        true,
        false,
        true
    ],
    "dominance_relations": [
        {
            "dominated_index": 0,
            "dominated_alternative": "A",
            "dominator_index": 4,
            "dominator_alternative": "E"
        }
    ],
    "comparison_stats": {
        "alternative_pair_comparisons": 25,
        "dimension_comparisons_better_or_equal": 75,
        "dimension_comparisons_strictly_better": 75,
        "total_dimension_comparisons": 150,
        "alternatives_evaluated": 6
    },
    "comparison_t

## **02. Cálculo de Matriz Normalizada**

### **02.1 Lectura de Matriz**

\begin{equation}
\mathbf{X} =
\left[
x_{ij}
\right]_{n \times m}
\end{equation}

\begin{equation}
\mathbf{X}^{P} =
\left[
x^{P}_{kj}
\right]_{p \times m}
\end{equation}

\begin{equation}
\mathbf{R}^{P} =
\left[
r^{P}_{kj}
\right]_{p \times m}
\end{equation}

\begin{equation}
r^{P}_{kj} =
\begin{cases}
\dfrac{x^{P}_{kj} - \min\limits_{k \in \mathcal{P}} x^{P}_{kj}}
{\max\limits_{k \in \mathcal{P}} x^{P}_{kj} - \min\limits_{k \in \mathcal{P}} x^{P}_{kj}},
& \text{si el criterio } j \text{ es de maximización}
\\[14pt]
\dfrac{\max\limits_{k \in \mathcal{P}} x^{P}_{kj} - x^{P}_{kj}}
{\max\limits_{k \in \mathcal{P}} x^{P}_{kj} - \min\limits_{k \in \mathcal{P}} x^{P}_{kj}},
& \text{si el criterio } j \text{ es de minimización}
\end{cases}
\end{equation}

\begin{equation}
0 \leq r^{P}_{kj} \leq 1
\end{equation}

\begin{equation}
r^{P}_{kj} \rightarrow 1
\quad \Longrightarrow \quad
\text{mejor desempeño relativo de la alternativa } k \text{ en el criterio } j
\end{equation}

\begin{equation}
\Delta_j =
\max\limits_{k \in \mathcal{P}} x^{P}_{kj}
-
\min\limits_{k \in \mathcal{P}} x^{P}_{kj}
\end{equation}

\begin{equation}
r^{P}_{kj} =
\begin{cases}
1,
& \text{si } \Delta_j = 0
\\[10pt]
\dfrac{x^{P}_{kj} - \min\limits_{k \in \mathcal{P}} x^{P}_{kj}}
{\Delta_j},
& \text{si } \Delta_j > 0 \text{ y } j \text{ es de maximización}
\\[14pt]
\dfrac{\max\limits_{k \in \mathcal{P}} x^{P}_{kj} - x^{P}_{kj}}
{\Delta_j},
& \text{si } \Delta_j > 0 \text{ y } j \text{ es de minimización}
\end{cases}
\end{equation}

\begin{align}
\mathcal{P} &= \text{ParetoSolver}(\mathbf{X}, \mathbf{d}) \\[6pt]
\mathbf{X}^{P} &= \mathbf{X}[\mathcal{P}, :] \\[6pt]
\mathbf{R}^{P} &= \text{Normalize}(\mathbf{X}^{P}, \mathbf{d})
\end{align}

\begin{equation}
\mathbf{d} =
[d_1, d_2, \dots, d_m],
\qquad
d_j \in \{\min, \max\}
\end{equation}

\begin{equation}
\left(
\mathbf{X},
\mathcal{P},
\mathbf{d}
\right)
\end{equation}

\begin{equation}
\mathbf{R}^{P}
\end{equation}

Sea la matriz de decisión del subconjunto no dominado:

\begin{equation}
\mathbf{X}^{P} =
\left[
x^{P}_{ij}
\right]_{p \times m}
\end{equation}

donde \(p\) es el número de alternativas no dominadas, \(m\) es el número de criterios, \(x^{P}_{ij}\) es el valor de la alternativa no dominada \(i\) en el criterio \(j\), y \(d_j\) representa la dirección del criterio:

\begin{equation}
d_j \in \{\min, \max\}
\end{equation}

La matriz normalizada se denota como:

\begin{equation}
\mathbf{R}^{P} =
\left[
r^{P}_{ij}
\right]_{p \times m}
\end{equation}

#### **02.1.1 Directional Min-Max Normalization**

\begin{equation}
r^{P}_{ij} =
\begin{cases}
\dfrac{x^{P}_{ij} - \min\limits_i x^{P}_{ij}}
{\max\limits_i x^{P}_{ij} - \min\limits_i x^{P}_{ij}},
& \text{si } d_j = \max
\\[14pt]
\dfrac{\max\limits_i x^{P}_{ij} - x^{P}_{ij}}
{\max\limits_i x^{P}_{ij} - \min\limits_i x^{P}_{ij}},
& \text{si } d_j = \min
\end{cases}
\end{equation}

\begin{equation}
\Delta_j =
\max\limits_i x^{P}_{ij}
-
\min\limits_i x^{P}_{ij}
\end{equation}

\begin{equation}
r^{P}_{ij} =
\begin{cases}
1,
& \text{si } \Delta_j = 0
\\[10pt]
\dfrac{x^{P}_{ij} - \min\limits_i x^{P}_{ij}}
{\Delta_j},
& \text{si } \Delta_j > 0 \text{ y } d_j = \max
\\[14pt]
\dfrac{\max\limits_i x^{P}_{ij} - x^{P}_{ij}}
{\Delta_j},
& \text{si } \Delta_j > 0 \text{ y } d_j = \min
\end{cases}
\end{equation}

#### **02.1.2 Vector Normalization**

Esta es la normalización vectorial estándar. No invierte los criterios de costo; solo elimina el efecto de escala.

\begin{equation}
r^{P}_{ij}
=
\dfrac{x^{P}_{ij}}
{\sqrt{\sum\limits_{i=1}^{p}
\left(x^{P}_{ij}\right)^2}}
\end{equation}

\begin{equation}
L_j =
\sqrt{\sum\limits_{i=1}^{p}
\left(x^{P}_{ij}\right)^2}
\end{equation}

\begin{equation}
r^{P}_{ij}
=
\begin{cases}
0,
& \text{si } L_j = 0
\\[10pt]
\dfrac{x^{P}_{ij}}{L_j},
& \text{si } L_j > 0
\end{cases}
\end{equation}

#### **02.1.3 Directional Vector Normalization**

Esta versión adapta la normalización vectorial para que todos los criterios queden orientados como beneficio, es decir, “mayor es mejor”.

\begin{equation}
L_j =
\sqrt{\sum\limits_{i=1}^{p}
\left(x^{P}_{ij}\right)^2}
\end{equation}

\begin{equation}
r^{P}_{ij} =
\begin{cases}
\dfrac{x^{P}_{ij}}{L_j},
& \text{si } d_j = \max
\\[14pt]
1 - \dfrac{x^{P}_{ij}}{L_j},
& \text{si } d_j = \min
\end{cases}
\end{equation}

\begin{equation}
r^{P}_{ij} =
\begin{cases}
0,
& \text{si } L_j = 0
\\[10pt]
\dfrac{x^{P}_{ij}}{L_j},
& \text{si } L_j > 0 \text{ y } d_j = \max
\\[14pt]
1 - \dfrac{x^{P}_{ij}}{L_j},
& \text{si } L_j > 0 \text{ y } d_j = \min
\end{cases}
\end{equation}

#### **02.1.4 Sum Normalization**

Normalización por suma estándar:

\begin{equation}
r^{P}_{ij}
=
\dfrac{x^{P}_{ij}}
{\sum\limits_{i=1}^{p} x^{P}_{ij}}
\end{equation}

\begin{equation}
S_j =
\sum\limits_{i=1}^{p} x^{P}_{ij}
\end{equation}

\begin{equation}
r^{P}_{ij}
=
\begin{cases}
0,
& \text{si } S_j = 0
\\[10pt]
\dfrac{x^{P}_{ij}}{S_j},
& \text{si } S_j \neq 0
\end{cases}
\end{equation}

#### **02.1.5 Z-Score Normalization**

La normalización Z-score transforma cada criterio en términos de desviaciones estándar respecto a su media.

\begin{equation}
r^{P}_{ij}
=
\dfrac{x^{P}_{ij} - \mu_j}
{\sigma_j}
\end{equation}

\begin{equation}
\mu_j =
\dfrac{1}{p}
\sum\limits_{i=1}^{p}
x^{P}_{ij}
\end{equation}

\begin{equation}
\sigma_j =
\sqrt{
\dfrac{1}{p}
\sum\limits_{i=1}^{p}
\left(
x^{P}_{ij} - \mu_j
\right)^2
}
\end{equation}

\begin{equation}
r^{P}_{ij}
=
\begin{cases}
0,
& \text{si } \sigma_j = 0
\\[10pt]
\dfrac{x^{P}_{ij} - \mu_j}{\sigma_j},
& \text{si } \sigma_j > 0
\end{cases}
\end{equation}

\begin{align}
\mathbf{X}^{P} &= \mathbf{X}[\mathcal{P}, :] \\[6pt]
\mathbf{R}^{P} &= \text{Normalize}(\mathbf{X}^{P}, \mathbf{d}) \\[6pt]
d_j &\in \{\min, \max\}
\end{align}

\begin{equation}
\boxed{
r^{P}_{ij} =
\begin{cases}
\dfrac{x^{P}_{ij} - \min\limits_i x^{P}_{ij}}
{\max\limits_i x^{P}_{ij} - \min\limits_i x^{P}_{ij}},
& \text{si } d_j = \max
\\[14pt]
\dfrac{\max\limits_i x^{P}_{ij} - x^{P}_{ij}}
{\max\limits_i x^{P}_{ij} - \min\limits_i x^{P}_{ij}},
& \text{si } d_j = \min
\end{cases}
}
\end{equation}

### **02.2 Script para Normalizar**

In [16]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Any, Sequence

import numpy as np
import numpy.typing as npt


class Direction(str, Enum):
    MIN = "min"
    MAX = "max"


class NormalizationMethod(str, Enum):
    DIRECTIONAL_MINMAX = "directional_minmax"
    VECTOR = "vector"
    DIRECTIONAL_VECTOR = "directional_vector"
    SUM = "sum"
    ZSCORE = "zscore"


@dataclass(frozen=True, slots=True)
class NormalizationResult:
    method: NormalizationMethod
    pareto_indices: list[int]
    pareto_alternatives: list[str]
    dimensions: list[str]
    directions: list[Direction]
    normalized_dimension_indices: list[int]
    normalized_dimensions: list[str]
    preserved_dimension_indices: list[int]
    preserved_dimensions: list[str]
    original_pareto_matrix: npt.NDArray[np.float64]
    normalized_matrix: npt.NDArray[np.float64]

    def to_dict(self) -> dict[str, Any]:
        return {
            "method": self.method.value,
            "pareto_indices": self.pareto_indices,
            "pareto_alternatives": self.pareto_alternatives,
            "dimensions": self.dimensions,
            "directions": [direction.value for direction in self.directions],
            "normalized_dimension_indices": self.normalized_dimension_indices,
            "normalized_dimensions": self.normalized_dimensions,
            "preserved_dimension_indices": self.preserved_dimension_indices,
            "preserved_dimensions": self.preserved_dimensions,
            "original_pareto_matrix": self.original_pareto_matrix.tolist(),
            "normalized_matrix": self.normalized_matrix.tolist(),
        }


class NonDominatedNormalizer:
    """
    Normalizes the non-dominated subset of a decision matrix.

    By default, all dimensions are normalized. However, the user can provide
    a subset of dimension names to normalize. Non-selected dimensions are
    preserved with their original values.
    """

    def __init__(
        self,
        matrix: Sequence[Sequence[float]] | npt.NDArray[np.float64],
        pareto_indices: Sequence[int],
        directions: Sequence[str | Direction],
        *,
        dimensions: Sequence[str] | None = None,
        alternatives: Sequence[str] | None = None,
        epsilon: float = 1e-12,
    ) -> None:
        self._matrix = np.asarray(matrix, dtype=np.float64)
        self._pareto_indices = [int(index) for index in pareto_indices]
        self._directions = [self._parse_direction(direction) for direction in directions]
        self._epsilon = float(epsilon)

        self._validate_base_inputs()

        self._n_alternatives: int = self._matrix.shape[0]
        self._n_dimensions: int = self._matrix.shape[1]

        self._dimensions = (
            list(dimensions)
            if dimensions is not None
            else [f"C{j + 1}" for j in range(self._n_dimensions)]
        )

        self._alternatives = (
            [str(alternative) for alternative in alternatives]
            if alternatives is not None
            else [f"A{i}" for i in range(self._n_alternatives)]
        )

        self._validate_metadata()

        self._pareto_matrix = self._matrix[self._pareto_indices, :]
        self._pareto_alternatives = [
            self._alternatives[index]
            for index in self._pareto_indices
        ]

    @staticmethod
    def _parse_direction(direction: str | Direction) -> Direction:
        if isinstance(direction, Direction):
            return direction

        normalized = str(direction).strip().lower()

        aliases = {
            "min": Direction.MIN,
            "minimize": Direction.MIN,
            "minimizar": Direction.MIN,
            "minimum": Direction.MIN,
            "cost": Direction.MIN,
            "costo": Direction.MIN,
            "max": Direction.MAX,
            "maximize": Direction.MAX,
            "maximizar": Direction.MAX,
            "maximum": Direction.MAX,
            "benefit": Direction.MAX,
            "beneficio": Direction.MAX,
        }

        if normalized not in aliases:
            raise ValueError(
                f"Invalid direction '{direction}'. "
                "Use 'min'/'max' or equivalent aliases."
            )

        return aliases[normalized]

    @staticmethod
    def _parse_method(method: str | NormalizationMethod) -> NormalizationMethod:
        if isinstance(method, NormalizationMethod):
            return method

        normalized = str(method).strip().lower()

        try:
            return NormalizationMethod(normalized)
        except ValueError as exc:
            valid_methods = [method.value for method in NormalizationMethod]
            raise ValueError(
                f"Invalid normalization method '{method}'. "
                f"Valid methods are: {valid_methods}."
            ) from exc

    def _validate_base_inputs(self) -> None:
        if self._matrix.ndim != 2:
            raise ValueError("matrix must be a two-dimensional array.")

        if self._matrix.size == 0:
            raise ValueError("matrix cannot be empty.")

        if not np.all(np.isfinite(self._matrix)):
            raise ValueError("matrix contains NaN or infinite values.")

        if not self._pareto_indices:
            raise ValueError("pareto_indices cannot be empty.")

        if len(set(self._pareto_indices)) != len(self._pareto_indices):
            raise ValueError("pareto_indices must be unique.")

        n_alternatives, n_dimensions = self._matrix.shape

        for index in self._pareto_indices:
            if index < 0 or index >= n_alternatives:
                raise IndexError(
                    f"Pareto index {index} is out of bounds for "
                    f"{n_alternatives} alternatives."
                )

        if len(self._directions) != n_dimensions:
            raise ValueError(
                "The number of directions must match the number of matrix columns. "
                f"Expected {n_dimensions}, got {len(self._directions)}."
            )

        if self._epsilon <= 0:
            raise ValueError("epsilon must be greater than zero.")

    def _validate_metadata(self) -> None:
        if len(self._dimensions) != self._n_dimensions:
            raise ValueError(
                "The number of dimensions must match the number of matrix columns. "
                f"Expected {self._n_dimensions}, got {len(self._dimensions)}."
            )

        if len(set(self._dimensions)) != len(self._dimensions):
            raise ValueError("Dimension names must be unique.")

        if len(self._alternatives) != self._n_alternatives:
            raise ValueError(
                "The number of alternatives must match the number of matrix rows. "
                f"Expected {self._n_alternatives}, got {len(self._alternatives)}."
            )

        if len(set(self._alternatives)) != len(self._alternatives):
            raise ValueError("Alternative identifiers must be unique.")

    def _resolve_dimensions_to_normalize(
        self,
        dimensions_to_normalize: Sequence[str] | None,
    ) -> tuple[list[int], list[str], list[int], list[str]]:
        """
        Resolves user-selected dimension names into column indices.

        If dimensions_to_normalize is None, all dimensions are normalized.
        """
        if dimensions_to_normalize is None:
            normalized_indices = list(range(self._n_dimensions))
        else:
            requested_dimensions = [str(name) for name in dimensions_to_normalize]

            if not requested_dimensions:
                raise ValueError(
                    "dimensions_to_normalize cannot be empty. "
                    "Use None to normalize all dimensions."
                )

            duplicated = {
                name
                for name in requested_dimensions
                if requested_dimensions.count(name) > 1
            }

            if duplicated:
                raise ValueError(
                    f"Duplicated dimensions in dimensions_to_normalize: {duplicated}"
                )

            dimension_to_index = {
                dimension: index
                for index, dimension in enumerate(self._dimensions)
            }

            missing = [
                dimension
                for dimension in requested_dimensions
                if dimension not in dimension_to_index
            ]

            if missing:
                raise ValueError(
                    f"Unknown dimensions requested for normalization: {missing}. "
                    f"Available dimensions are: {self._dimensions}."
                )

            normalized_indices = [
                dimension_to_index[dimension]
                for dimension in requested_dimensions
            ]

        preserved_indices = [
            index
            for index in range(self._n_dimensions)
            if index not in set(normalized_indices)
        ]

        normalized_dimensions = [
            self._dimensions[index]
            for index in normalized_indices
        ]

        preserved_dimensions = [
            self._dimensions[index]
            for index in preserved_indices
        ]

        return (
            normalized_indices,
            normalized_dimensions,
            preserved_indices,
            preserved_dimensions,
        )

    def normalize(
        self,
        method: str | NormalizationMethod,
        *,
        dimensions_to_normalize: Sequence[str] | None = None,
    ) -> NormalizationResult:
        selected_method = self._parse_method(method)

        (
            normalized_indices,
            normalized_dimensions,
            preserved_indices,
            preserved_dimensions,
        ) = self._resolve_dimensions_to_normalize(dimensions_to_normalize)

        if selected_method == NormalizationMethod.DIRECTIONAL_MINMAX:
            fully_normalized = self._directional_minmax()

        elif selected_method == NormalizationMethod.VECTOR:
            fully_normalized = self._vector()

        elif selected_method == NormalizationMethod.DIRECTIONAL_VECTOR:
            fully_normalized = self._directional_vector()

        elif selected_method == NormalizationMethod.SUM:
            fully_normalized = self._sum()

        elif selected_method == NormalizationMethod.ZSCORE:
            fully_normalized = self._zscore()

        else:
            raise NotImplementedError(
                f"Normalization method '{selected_method.value}' is not implemented."
            )

        normalized_matrix = self._pareto_matrix.copy()
        normalized_matrix[:, normalized_indices] = fully_normalized[:, normalized_indices]

        return NormalizationResult(
            method=selected_method,
            pareto_indices=self._pareto_indices.copy(),
            pareto_alternatives=self._pareto_alternatives.copy(),
            dimensions=self._dimensions.copy(),
            directions=self._directions.copy(),
            normalized_dimension_indices=normalized_indices,
            normalized_dimensions=normalized_dimensions,
            preserved_dimension_indices=preserved_indices,
            preserved_dimensions=preserved_dimensions,
            original_pareto_matrix=self._pareto_matrix.copy(),
            normalized_matrix=normalized_matrix,
        )

    def normalize_as_dict(
        self,
        method: str | NormalizationMethod,
        *,
        dimensions_to_normalize: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        return self.normalize(
            method,
            dimensions_to_normalize=dimensions_to_normalize,
        ).to_dict()

    def normalize_all(
        self,
        methods: Sequence[str | NormalizationMethod] | None = None,
        *,
        dimensions_to_normalize: Sequence[str] | None = None,
    ) -> dict[NormalizationMethod, NormalizationResult]:
        if methods is None:
            selected_methods = list(NormalizationMethod)
        else:
            selected_methods = [self._parse_method(method) for method in methods]

        return {
            method: self.normalize(
                method,
                dimensions_to_normalize=dimensions_to_normalize,
            )
            for method in selected_methods
        }

    def normalize_all_as_dict(
        self,
        methods: Sequence[str | NormalizationMethod] | None = None,
        *,
        dimensions_to_normalize: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        results = self.normalize_all(
            methods,
            dimensions_to_normalize=dimensions_to_normalize,
        )

        return {
            method.value: result.to_dict()
            for method, result in results.items()
        }

    def _directional_minmax(self) -> npt.NDArray[np.float64]:
        x = self._pareto_matrix
        mins = np.min(x, axis=0)
        maxs = np.max(x, axis=0)
        ranges = maxs - mins

        normalized = np.zeros_like(x, dtype=np.float64)

        for j, direction in enumerate(self._directions):
            if ranges[j] <= self._epsilon:
                normalized[:, j] = 1.0
                continue

            if direction == Direction.MAX:
                normalized[:, j] = (x[:, j] - mins[j]) / ranges[j]
            else:
                normalized[:, j] = (maxs[j] - x[:, j]) / ranges[j]

        return normalized

    def _vector(self) -> npt.NDArray[np.float64]:
        x = self._pareto_matrix
        norms = np.sqrt(np.sum(np.square(x), axis=0))

        normalized = np.zeros_like(x, dtype=np.float64)

        valid = norms > self._epsilon
        normalized[:, valid] = x[:, valid] / norms[valid]

        return normalized

    def _directional_vector(self) -> npt.NDArray[np.float64]:
        x = self._pareto_matrix
        norms = np.sqrt(np.sum(np.square(x), axis=0))

        normalized = np.zeros_like(x, dtype=np.float64)

        for j, direction in enumerate(self._directions):
            if norms[j] <= self._epsilon:
                normalized[:, j] = 0.0
                continue

            vector_values = x[:, j] / norms[j]

            if direction == Direction.MAX:
                normalized[:, j] = vector_values
            else:
                normalized[:, j] = 1.0 - vector_values

        return normalized

    def _sum(self) -> npt.NDArray[np.float64]:
        x = self._pareto_matrix
        normalized = np.zeros_like(x, dtype=np.float64)

        for j, direction in enumerate(self._directions):
            column = x[:, j]

            if direction == Direction.MAX:
                denominator = np.sum(column)

                if abs(denominator) <= self._epsilon:
                    normalized[:, j] = 0.0
                else:
                    normalized[:, j] = column / denominator

            else:
                if np.any(column <= self._epsilon):
                    raise ValueError(
                        f"Sum normalization for cost criterion "
                        f"'{self._dimensions[j]}' requires strictly positive values."
                    )

                reciprocal = 1.0 / column
                denominator = np.sum(reciprocal)

                if denominator <= self._epsilon:
                    normalized[:, j] = 0.0
                else:
                    normalized[:, j] = reciprocal / denominator

        return normalized

    def _zscore(self) -> npt.NDArray[np.float64]:
        x = self._pareto_matrix
        means = np.mean(x, axis=0)
        stds = np.std(x, axis=0, ddof=0)

        normalized = np.zeros_like(x, dtype=np.float64)

        for j, direction in enumerate(self._directions):
            if stds[j] <= self._epsilon:
                normalized[:, j] = 0.0
                continue

            if direction == Direction.MAX:
                normalized[:, j] = (x[:, j] - means[j]) / stds[j]
            else:
                normalized[:, j] = (means[j] - x[:, j]) / stds[j]

        return normalized

#### **02.2.1 Caso de Uso**

In [17]:
def main() -> None:
    matrix = [
        [170, 0.6393, 0.2300],
        [195, 0.7090, 0.3000],
        [128, 0.6716, 0.7000],
        [188, 0.6138, 0.8000],
        [115, 0.6442, 0.1000],
        [135, 0.5642, 0.4000],
    ]

    pareto_indices = [1, 2, 4]

    dimensions = ["OMOC", "OMOE", "OMOR"]
    directions = ["min", "max", "min"]
    alternatives = ["A", "B", "C", "D", "E", "F"]

    normalizer = NonDominatedNormalizer(
        matrix=matrix,
        pareto_indices=pareto_indices,
        directions=directions,
        dimensions=dimensions,
        alternatives=alternatives,
        epsilon=1e-12,
    )

    result = normalizer.normalize(
    method="directional_vector",
    dimensions_to_normalize=["OMOC", "OMOR"],
)

    print(result.normalized_matrix)
    print(result.to_dict())

    all_results = normalizer.normalize_all_as_dict()

    for method_name, output in all_results.items():
        print(f"\nMethod: {method_name}")
        print(output["normalized_matrix"])

if __name__ == "__main__":
    main()

[[0.25018854 0.709      0.60943327]
 [0.50781607 0.6716     0.08867762]
 [0.5578035  0.6442     0.86981109]]
{'method': 'directional_vector', 'pareto_indices': [1, 2, 4], 'pareto_alternatives': ['B', 'C', 'E'], 'dimensions': ['OMOC', 'OMOE', 'OMOR'], 'directions': ['min', 'max', 'min'], 'normalized_dimension_indices': [0, 2], 'normalized_dimensions': ['OMOC', 'OMOR'], 'preserved_dimension_indices': [1], 'preserved_dimensions': ['OMOE'], 'original_pareto_matrix': [[195.0, 0.709, 0.3], [128.0, 0.6716, 0.7], [115.0, 0.6442, 0.1]], 'normalized_matrix': [[0.25018853835020005, 0.709, 0.6094332670575284], [0.5078160661991057, 0.6716, 0.088677623134233], [0.557803496975759, 0.6442, 0.8698110890191761]]}

Method: directional_minmax
[[0.0, 1.0, 0.6666666666666666], [0.8375, 0.4228395061728394, 0.0], [1.0, 0.0, 1.0]]

Method: vector
[[0.7498114616498, 0.606022485338187, 0.3905667329424716], [0.4921839338008943, 0.5740545855474279, 0.911322376865767], [0.44219650302424096, 0.5506342525456419, 0.13

## **03. Importación de Métodos MADM**